In [1]:
import pandas as pd

In [8]:
notes = pd.read_csv('data/NOTEEVENTS.csv')
diagnoses = pd.read_csv('data/DIAGNOSES_ICD.csv')
dictionary = pd.read_csv('data/D_ICD_DIAGNOSES.csv')

In [9]:
notes.shape

(482770, 11)

In [10]:
notes.head()

,ROW_ID,SUBJECT_ID,HADM_ID,CHARTDATE,CHARTTIME,STORETIME,CATEGORY,DESCRIPTION,CGID,ISERROR,TEXT
0,1678764,2,163353.0,2138-07-17,2138-07-17 22:51:00,2138-07-17 23:12:00,Nursing/other,Report,16929.0,NaN,Neonatology Attending Triage Note\n\nBaby [**N...
1,1678765,2,163353.0,2138-07-17,2138-07-17 23:08:00,2138-07-17 23:18:00,Nursing/other,Report,17774.0,NaN,Nursing Transfer note\n\n\nPt admitted to NICU...
2,272794,3,NaN,2101-10-06,NaN,NaN,ECG,Report,NaN,NaN,Sinus rhythm\nInferior/lateral ST-T changes ar...
3,769224,3,145834.0,2101-10-26,2101-10-26 06:01:00,NaN,Radiology,CHEST (PORTABLE AP),NaN,NaN,[**2101-10-26**] 6:01 AM\n CHEST (PORTABLE AP)...
4,272793,3,NaN,2101-10-11,NaN,NaN,ECG,Report,NaN,NaN,Sinus rhythm\nA-V delay\nNonspecific inferior ...


In [11]:
notes_filt = notes[notes['CATEGORY'] == 'Discharge summary'][['HADM_ID', 'TEXT']]

In [12]:
notes_filt


,HADM_ID,TEXT
30,145834.0,Admission Date: [**2101-10-20**] Discharg...
102,185777.0,Admission Date: [**2191-3-16**] Discharge...
116,107064.0,Admission Date: [**2175-5-30**] Dischar...
158,150750.0,"Name: [**Known lastname 10050**], [**Known fi..."
166,150750.0,Admission Date: [**2149-11-9**] Dischar...
...,...,...
482313,164518.0,Admission Date: [**2186-8-14**] Discharge...
482315,182655.0,Admission Date: [**2184-3-22**] Discharge...
482671,117042.0,Admission Date: [**2119-5-27**] Dischar...
482672,117042.0,Admission Date: [**2119-5-27**] Dischar...


In [13]:
df_combined = pd.merge(notes_filt, diagnoses[['HADM_ID', 'ICD9_CODE']], on='HADM_ID')

In [14]:
df_combined.dropna(subset=['TEXT', 'ICD9_CODE'], inplace=True)

In [15]:
df_combined.shape[0]

128517

In [16]:
df_combined 

,HADM_ID,TEXT,ICD9_CODE
0,145834.0,Admission Date: [**2101-10-20**] Discharg...,2639
1,145834.0,Admission Date: [**2101-10-20**] Discharg...,6826
2,145834.0,Admission Date: [**2101-10-20**] Discharg...,4280
3,145834.0,Admission Date: [**2101-10-20**] Discharg...,41071
4,145834.0,Admission Date: [**2101-10-20**] Discharg...,4254
...,...,...,...
128512,181987.0,Unit No: [**Numeric Identifier 62050**]\nAdmi...,7757
128513,181987.0,Unit No: [**Numeric Identifier 62050**]\nAdmi...,7742
128514,181987.0,Unit No: [**Numeric Identifier 62050**]\nAdmi...,769
128515,181987.0,Unit No: [**Numeric Identifier 62050**]\nAdmi...,V3101


In [17]:
import re

In [18]:
def clean_medical_text(text):
    text = text.lower()
    text = re.sub(r'\[\*\*.*?\*\*\]', '', text) # Remove de-identified brackets
    text = re.sub(r'[^a-zA-Z\s]', '', text)     # Remove numbers/special chars
    text = re.sub(r'\s+', ' ', text).strip()    # Remove extra whitespace
    return text

df_combined['CLEAN_TEXT'] = df_combined['TEXT'].apply(clean_medical_text)

In [ ]:
# Get the top 50 codes
top_500_codes = df_combined['ICD9_CODE'].value_counts().nlargest(500).index
df_final = df_combined[df_combined['ICD9_CODE'].isin(top_50_codes)]

print(f"🚀 Final dataset ready with {len(top_500_codes)} unique medical categories!")

🚀 Final dataset ready with 500 unique medical categories!


In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [23]:
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english', ngram_range=(1, 2))
X = vectorizer.fit_transform(df_final['CLEAN_TEXT'])
y = df_final['ICD9_CODE']

In [24]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

In [25]:
le = LabelEncoder()
y_encoded = le.fit_transform(df_final['ICD9_CODE'])

In [26]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42
)

In [27]:
from xgboost import XGBClassifier

In [ ]:
model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    objective='multi:softprob',
    tree_method='hist',  # Use 'gpu_hist' if you have an NVIDIA GPU!
    num_class=500,
    n_jobs=-1            # Use all your CPU cores
)

print("🚀 Training started... Go grab a coffee, this is a big one!")
model.fit(X_train, y_train)

🚀 Training started... Go grab a coffee, this is a big one!
